# Generate LAION-Art AMP perturbations

Generate one LLaVA AMP adversarial image for each source-target row in the prepared attack-set manifest. Existing valid outputs are skipped so interrupted runs can resume.

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import LlavaForConditionalGeneration

MANIFEST_PATH = Path("dataset/laion_art/attack_set/manifest.csv")
ADV_DIR = MANIFEST_PATH.parent / "adv"
RESULTS_PATH = MANIFEST_PATH.parent / "attack_results.csv"

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
BUDGET = 16 / 255
NUM_OPTIMIZATION_STEPS = 4000
INITIAL_LR = 0.005
MAX_ATTACKS = None  # Set to 1 for a quick test.

DTYPE = torch.float16
LLAVA_IMAGE_SIZE = 336


In [ ]:
# Load the same LLaVA vision tower and preprocessing used by attack_llava.py once.
llava_model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)
vision_tower = llava_model.vision_tower.cuda()

to_tensor = transforms.ToTensor()
transform = transforms.Compose(
    [
        transforms.Resize(
            (LLAVA_IMAGE_SIZE, LLAVA_IMAGE_SIZE),
            interpolation=transforms.InterpolationMode.BICUBIC,
        ),
        transforms.Normalize(
            (0.48145466, 0.4578275, 0.40821073),
            (0.26862954, 0.26130258, 0.27577711),
        ),
    ]
)


In [ ]:
def generate_perturbation(source_path, target_path, output_path):
    source_img = Image.open(source_path)
    source_tensor = to_tensor(source_img).to("cuda", DTYPE)
    modifier = torch.clone(source_tensor) * 0.1

    target_img = Image.open(target_path)
    target_tensor = to_tensor(target_img).to("cuda", DTYPE)
    target_feature = vision_tower(
        torch.stack([transform(target_tensor)]), output_hidden_states=True
    ).hidden_states[-2]

    for i in tqdm(
        range(NUM_OPTIMIZATION_STEPS),
        desc="[llava]: generating perturbation",
        leave=False,
    ):
        # Keep the original linear schedule and sign-gradient update unchanged.
        alpha = (
            INITIAL_LR
            - (INITIAL_LR - INITIAL_LR / 100) / NUM_OPTIMIZATION_STEPS * i
        )
        modifier.requires_grad_(True)

        adv_tensor = torch.clamp(modifier + source_tensor, 0, 1)
        adv_feature = vision_tower(
            torch.stack([transform(adv_tensor)]), output_hidden_states=True
        ).hidden_states[-2]

        loss = (adv_feature - target_feature).norm()
        grad = torch.autograd.grad(loss, modifier)[0]
        modifier = modifier.detach()
        modifier = modifier - torch.sign(grad) * alpha
        modifier = torch.clamp(modifier, min=-BUDGET, max=BUDGET)

    adv_image = source_tensor + modifier
    adv_image = torch.clamp(adv_image, 0.0, 1.0)
    adv_img = transforms.ToPILImage()(adv_image.to(torch.float16))
    adv_img.save(output_path)


def is_valid_image(path):
    if not path.is_file():
        return False
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except (OSError, SyntaxError):
        return False


In [ ]:
# Process the selected manifest rows and record every outcome without changing the manifest.
manifest = pd.read_csv(MANIFEST_PATH, dtype={"sample_id": str})
required_columns = {"sample_id", "source_path", "target_path"}
missing_columns = required_columns - set(manifest.columns)
if missing_columns:
    raise ValueError(f"Missing columns in {MANIFEST_PATH}: {sorted(missing_columns)}")
if manifest["sample_id"].duplicated().any():
    raise ValueError(f"Duplicate sample IDs in {MANIFEST_PATH}")

rows = manifest if MAX_ATTACKS is None else manifest.head(MAX_ATTACKS)
ADV_DIR.mkdir(parents=True, exist_ok=True)
results = []

for row in tqdm(rows.itertuples(index=False), total=len(rows), desc="AMP attacks"):
    source_path = Path(row.source_path)
    target_path = Path(row.target_path)
    adv_path = ADV_DIR / f"{row.sample_id}.png"
    result = {
        "sample_id": row.sample_id,
        "source_path": source_path.as_posix(),
        "target_path": target_path.as_posix(),
        "adv_path": adv_path.as_posix(),
    }

    if is_valid_image(adv_path):
        result["status"] = "skipped"
    else:
        try:
            generate_perturbation(source_path, target_path, adv_path)
            result["status"] = "completed"
        except Exception as error:
            result["status"] = "failed"
            result["error"] = f"{type(error).__name__}: {error}"

    results.append(result)
    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

result_table = pd.DataFrame(
    results,
    columns=["sample_id", "source_path", "target_path", "adv_path", "status", "error"],
)
counts = result_table["status"].value_counts()
print(
    "Summary: "
    f"completed={counts.get('completed', 0)}, "
    f"skipped={counts.get('skipped', 0)}, "
    f"failed={counts.get('failed', 0)}"
)
result_table
